In [2]:
from bs4 import BeautifulSoup
import requests
import re

headers = {"User-Agent": "Mozilla5.0"}
base_url = 'https://en.wikipedia.org'
url = 'https://en.wikipedia.org/wiki/List_of_highest-grossing_films'
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Поиск заголовка
content = soup.find('div', id='bodyContent')
print(len(content))
table = content.find('table')
hrefs = dict()
data = dict()
if table:
    rows = table.find_all('tr')
    for row in rows:
        cols = row.find_all(['td', 'th'])
        cols = [elem.text.strip() for elem in cols]
        links = row.find_all('a', href=True)
        if len(links) > 0:
            for link in links:
                link = link.get('href')
                if 'wiki' in link:
                    hrefs[cols[2]] = link
        data[cols[2]] = [cols[4]]

data[list(data.keys())[0]].extend(['language', 'directors', 'production companies', 'countries', 'box office'])

for key in hrefs:

    link = hrefs[key]

    languages = []
    directors = []
    countries = []
    box_office = []
    companies = []
    if link:
        response2 = requests.get(base_url + link, headers=headers)
        soup2 = BeautifulSoup(response2.text, 'html.parser')
        content = soup2.find_all('div', {'class':'mw-content-ltr mw-parser-output'})[0]
        table_mv = content.find('table')
        rows = table_mv.find_all('tr')
        for row in rows:
            cols = row.find_all(['td', 'th'])
            cols = [elem.text.strip() for elem in cols]
            if 'Directed by' in cols:
                directors.append(cols[1:])
            elif 'Countries' in cols or 'Country' in cols:
                countries.append(cols[1].split('\n')[0].split('[')[0])
            elif 'Language' in cols:
                languages.append(cols[1:])
            elif 'Productioncompanies' in cols or 'Productioncompany' in cols:
                prod_comp = cols[1].split('\n')
                for company in prod_comp:
                    company = company.split('[')[0]
                    companies.append(company)
            elif 'Box office' in cols:
                cost = cols[1].split('[')[0].strip()
                if 'billion' in cost:
                    mult = 1e9
                elif 'million' in cost:
                    mult = 1e6
                else:
                    print("Error!!!")
                cost = re.sub(r'[^\d.,-]', '', cost)
                box_office.append(float(cost) * mult)
        data[key].extend(languages)
        data[key].extend(directors)
        data[key].append(companies)
        data[key].append(countries[0])
        data[key].append(box_office[0])
for row in data:
    print(row, data[row], sep='\t')


9
Title	['Year', 'language', 'directors', 'production companies', 'countries', 'box office']
Avatar	['2009', ['English'], ['James Cameron'], ['20th Century Fox', 'Lightstorm Entertainment', 'Dune Entertainment', 'Ingenious Film Partners'], 'United States', 2924000000.0]
Avengers: Endgame	['2019', ['English'], ['Anthony RussoJoe Russo'], ['Marvel Studios'], 'United States', 2799000000.0]
Avatar: The Way of Water	['2022', ['English'], ['James Cameron'], ['20th Century Studios', 'Lightstorm Entertainment'], 'United States', 2334000000.0]
Titanic	['1997', ['English'], ['James Cameron'], ['Paramount Pictures', '20th Century Fox', 'Lightstorm Entertainment'], 'United States', 2264000000.0]
Ne Zha 2	['2025', ['Mandarin'], ['Jiaozi'], ['Chengdu Coco Cartoon', 'Beijing Enlight Media', 'Beijing Enlight Pictures', 'Chengdu Zizai Jingjie Culture Media', 'Beijing Coloroom Technology'], 'China', 2216000000.0]
Star Wars: The Force Awakens	['2015', ['English'], ['J. J. Abrams'], ['Lucasfilm Ltd.', 'Ba

In [13]:
import sqlite3

db_path = './films.db'

def create_table(cursor):
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS films (
        id INTEGER PRIMARY KEY,
        title TEXT NOT NULL,
        languages TEXT,
        release_year INTEGER,
        directors TEXT,
        production_companies TEXT,
        country TEXT, 
        box_office REAL
        )
        ''')


def fill_table(cursor, data):
    id = 1
    for key in data:
        if key == 'Title':
            continue
        info = data[key].copy()
        print(info)
        info[1] = '\n'.join(info[1])
        info[2] = '\n'.join(info[2])
        info[3] = '\n'.join(info[3])
        print(info)
        cursor.execute('''INSERT OR IGNORE INTO films (id, title, release_year, languages, directors, production_companies, country, box_office)
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)''', (id, key, info[0], info[1], info[2], info[3], info[4], info[5], ))
        id += 1


conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS films")

create_table(cursor)
fill_table(cursor, data)

conn.commit()
conn.close()

['2009', ['English'], ['James Cameron'], ['20th Century Fox', 'Lightstorm Entertainment', 'Dune Entertainment', 'Ingenious Film Partners'], 'United States', 2924000000.0]
['2009', 'English', 'James Cameron', '20th Century Fox\nLightstorm Entertainment\nDune Entertainment\nIngenious Film Partners', 'United States', 2924000000.0]
['2019', ['English'], ['Anthony RussoJoe Russo'], ['Marvel Studios'], 'United States', 2799000000.0]
['2019', 'English', 'Anthony RussoJoe Russo', 'Marvel Studios', 'United States', 2799000000.0]
['2022', ['English'], ['James Cameron'], ['20th Century Studios', 'Lightstorm Entertainment'], 'United States', 2334000000.0]
['2022', 'English', 'James Cameron', '20th Century Studios\nLightstorm Entertainment', 'United States', 2334000000.0]
['1997', ['English'], ['James Cameron'], ['Paramount Pictures', '20th Century Fox', 'Lightstorm Entertainment'], 'United States', 2264000000.0]
['1997', 'English', 'James Cameron', 'Paramount Pictures\n20th Century Fox\nLightstorm

Exporting database to the json

In [14]:
import sqlite3
import json

conn = sqlite3.connect('films.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM films")
columns = [description[0] for description in cursor.description]
rows = cursor.fetchall()

# Convert to list of dictionaries
results = [dict(zip(columns, row)) for row in rows]

# Save to a JSON file
with open('output.json', 'w') as f:
    json.dump(results, f, indent=4)

conn.close()